# Improved LSTM Forecasting

## 1. Import Libraries

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, Input, LSTM
from tensorflow.keras.models import Sequential

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_bitcoin_data
from src.preprocessing import prepare_daily_bitcoin_data
from src.metrics import mae, rmse, mape, smape

tf.random.set_seed(42)
np.random.seed(42)

## 2. Load Dataset

In [ ]:
data_path = PROJECT_ROOT / "data" / "bitcoin" / "btcusd_1-min_data.csv"

df_raw = load_bitcoin_data(data_path)
df_daily = prepare_daily_bitcoin_data(df_raw)
target = df_daily["Close"].dropna().asfreq("D")

df_daily.head()

## 3. Train-Test Split

In [ ]:
split_idx = int(len(target) * 0.8)

train = target.iloc[:split_idx]
test = target.iloc[split_idx:]
y_test = test.copy()

train.shape, test.shape

## 4. Data Scaling

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))

train_values = train.to_numpy().reshape(-1, 1)
test_values = test.to_numpy().reshape(-1, 1)

train_scaled = scaler.fit_transform(train_values)
test_scaled = scaler.transform(test_values)

scaler.feature_range

## 5. Sequence Creation

In [ ]:
LOOKBACK = 60


def create_sequences(values, lookback):
    X, y = [], []
    for i in range(lookback, len(values)):
        X.append(values[i - lookback : i])
        y.append(values[i])
    return np.array(X), np.array(y)


X_train, y_train = create_sequences(train_scaled, LOOKBACK)

combined_scaled = np.vstack([train_scaled[-LOOKBACK:], test_scaled])
X_test, y_test_scaled = create_sequences(combined_scaled, LOOKBACK)

X_train.shape, y_train.shape, X_test.shape, y_test.shape

## 6. Build Original LSTM Benchmark

In [ ]:
ORIGINAL_LOOKBACK = 30

X_train_original, y_train_original = create_sequences(train_scaled, ORIGINAL_LOOKBACK)
combined_scaled_original = np.vstack([train_scaled[-ORIGINAL_LOOKBACK:], test_scaled])
X_test_original, y_test_original_scaled = create_sequences(
    combined_scaled_original,
    ORIGINAL_LOOKBACK,
)

original_model = Sequential(
    [
        Input(shape=(ORIGINAL_LOOKBACK, 1)),
        LSTM(32),
        Dense(1),
    ]
)

original_model.compile(optimizer="adam", loss="mse")
original_model.summary()

## 7. Train Original LSTM Benchmark

In [ ]:
original_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
)

original_history = original_model.fit(
    X_train_original,
    y_train_original,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    callbacks=[original_early_stopping],
    shuffle=False,
)

original_predictions_scaled = original_model.predict(X_test_original)
original_predictions = scaler.inverse_transform(original_predictions_scaled).ravel()
original_lstm_forecast = pd.Series(
    original_predictions,
    index=test.index,
    name="Original LSTM",
)

## 8. Build Improved LSTM Model

In [ ]:
improved_model = Sequential(
    [
        Input(shape=(LOOKBACK, 1)),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(1),
    ]
)

improved_model.compile(optimizer="adam", loss="mse")
improved_model.summary()

## 9. Train Improved Model

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
)

history = improved_model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stopping],
    shuffle=False,
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history.history["loss"], label="Train Loss")
ax.plot(history.history["val_loss"], label="Validation Loss")
ax.set_title("Improved LSTM Training Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 10. Generate Improved Forecasts

In [ ]:
improved_predictions_scaled = improved_model.predict(X_test)
improved_predictions = scaler.inverse_transform(improved_predictions_scaled).ravel()
improved_lstm_forecast = pd.Series(
    improved_predictions,
    index=test.index,
    name="Improved LSTM",
)

fig, ax = plt.subplots(figsize=(12, 5))
train.tail(180).plot(ax=ax, label="Train")
test.plot(ax=ax, label="Test")
improved_lstm_forecast.plot(ax=ax, label="Improved LSTM Forecast")
ax.set_title("Bitcoin Close Price: Improved LSTM Forecast")
ax.set_xlabel("Date")
ax.set_ylabel("Close")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 11. Benchmarks

In [ ]:
naive_forecast = target.shift(1).reindex(test.index).rename("Naive")
moving_average_forecast = (
    target.shift(1)
    .rolling(window=7)
    .mean()
    .reindex(test.index)
    .rename("7-Day Moving Average")
)

benchmark_forecasts = {
    "Naive": naive_forecast,
    "7-Day Moving Average": moving_average_forecast,
    "Original LSTM": original_lstm_forecast,
    "Improved LSTM": improved_lstm_forecast,
}

## 12. Evaluation Metrics

In [ ]:
def evaluate_forecast(y_true, y_pred):
    return {
        "MAE": mae(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MAPE": mape(y_true, y_pred),
        "sMAPE": smape(y_true, y_pred),
    }


metrics_table = pd.DataFrame(
    [evaluate_forecast(y_test, forecast) for forecast in benchmark_forecasts.values()],
    index=benchmark_forecasts.keys(),
)

metrics_table.sort_values("RMSE")

## 13. Forecast Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
y_test.plot(ax=ax, label="Actual", linewidth=2)

for model_name, forecast in benchmark_forecasts.items():
    forecast.plot(ax=ax, label=model_name, alpha=0.85)

ax.set_title("Improved LSTM vs Benchmarks")
ax.set_xlabel("Date")
ax.set_ylabel("Close")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 14. Key Findings
- This improved LSTM uses a 60-day lookback window, two LSTM layers, dropout, and early stopping.
- The 20-epoch cap keeps this as a compact experiment before broader tuning.
- Naive and 7-day moving average forecasts remain essential benchmarks because Bitcoin prices are highly persistent.
- The original LSTM benchmark is trained in this notebook with a 30-day lookback and a single LSTM layer so the comparison is computed on the same test period.